# Maia Lite 0.3.0 — SFT v2.3 | Training Engine L4/A100

**Novo experimento independente:** base MaiaGPT v3.8, checkpoint 300.000; corpus auditado SFT v2.2 (sem regeneração); separador `\n`; contexto configurável (768 padrão, 1024 após teste de VRAM); LoRA rank 16 / alpha 32.

**Segurança operacional:** `RUN_TRAINING=False` inicialmente. Execute as células em ordem; confirme auditoria, amostras, teste de máscara e memória. Só então habilite treinamento. A saída usa diretório **v2.3 separado**, preservando integralmente o adaptador v2.2.

**Retomada:** checkpoint atômico somente em fronteiras de update completo, incluindo optimizer, scheduler, estados RNG e configuração. O percurso é reprodutível a partir do último checkpoint íntegro sob o mesmo hardware/ambiente, mas não se promete igualdade bit a bit entre GPUs ou kernels diferentes. Não retome de checkpoints parciais. 

**Limitações:** este notebook não certifica licenças nem revisão humana; exige aprovação anterior e hashes íntegros. Avaliação de loss não substitui benchmark de geração.

In [ ]:
import os, json, math, time, random, hashlib, gc, csv, sys, subprocess
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
try:
    from transformers import AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers'])
    from transformers import AutoTokenizer
from tqdm.auto import tqdm

SEED=20261118
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Selecione uma GPU L4 ou A100 no Colab.'
print('GPU:',torch.cuda.get_device_name(0),'| PyTorch:',torch.__version__)
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
except ImportError:pass
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive não montado.'

In [ ]:
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
PREP=ROOT/'datasets/sft_v2_2_aya'
TRAIN_FILE=PREP/'sft_v2_2_train.jsonl'
VAL_FILE=PREP/'sft_v2_2_validation.jsonl'
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
OUT=ROOT/'training_runs/maia_lite_0_3_0_sft_v2_3_lora_v1'
assert OUT.name != 'maia_lite_0_3_0_sft_v2_2_lora_v1'
for path in (TRAIN_FILE,VAL_FILE,BASE_CKPT):assert path.is_file(),f'Arquivo ausente: {path}'
assert TOKENIZER_DIR.is_dir(),f'Tokenizer ausente: {TOKENIZER_DIR}'
print('Treino:',TRAIN_FILE,'| Validação:',VAL_FILE)
print('Base:',BASE_CKPT,'| Saída v2.3:',OUT)

## Pré-requisito obrigatório: auditoria aprovada

Aprovação manual registrada no notebook de auditoria e hashes correspondentes são exigidos. Se qualquer arquivo mudar, refaça a auditoria.

In [ ]:
AUDIT=PREP/'audit_v2_2/audit_pass.json'
assert AUDIT.is_file(),f'Auditoria pendente: {AUDIT}'
audit=json.loads(AUDIT.read_text(encoding='utf8'))
def file_hash(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
assert audit['issues_count']==0 and audit['human_review_approved'] and audit['license_review_approved']
assert audit['train_sha256']==file_hash(TRAIN_FILE),'Treino alterado após auditoria'
assert audit['validation_sha256']==file_hash(VAL_FILE),'Validação alterada após auditoria'
print('Auditoria aprovada e hashes verificados')

## Parâmetros e proteção contra treinamento acidental

Primeira execução: `RUN_TRAINING=False`. Faça o smoke test de memória com `RUN_MEMORY_PROBE=True`; este teste executa um único forward/backward com LoRA sem `optimizer.step()` e limpa os gradientes. Na L4, comece em 768. A opção 1024 só deve ser usada se o teste couber com margem.

In [ ]:
RANK=16;ALPHA=32;LORA_DROPOUT=0.05
LR=8e-5;WEIGHT_DECAY=0.0
MAX_LENGTH=768  # mude para 1024 apenas após medir VRAM; requer novo OUT
SEPARATOR='\n' # delimitador auditado; NÃO altera os arquivos originais
EPOCHS=1;MICRO_BATCH=1;GRAD_ACCUM=16
MAX_GRAD_NORM=1.0;VALIDATE_EVERY=100;SAVE_EVERY=100
MAX_STEPS=None # para teste curto, ex. 10, use um OUT exclusivo e mantenha a configuração
RUN_MEMORY_PROBE=True
RUN_TRAINING=False  # habilite SOMENTE após revisar auditoria, cobertura e VRAM
assert MAX_LENGTH in (768,1024)
assert MAX_LENGTH<=2048 and MICRO_BATCH>=1 and GRAD_ACCUM>=1

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## Carregar checkpoint original (somente leitura)

`weights_only=False` é necessário porque o checkpoint confiável do próprio treinamento contém estados RNG e optimizer. **Nunca use essa opção com checkpoints de origem desconhecida.** A arquitetura é reproduzida diretamente do notebook original v3.8.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size,(len(tokenizer),cfg.vocab_size)
assert tokenizer.eos_token_id is not None,'Tokenizer sem EOS.'
print('EOS:',tokenizer.eos_token_id,'vocab:',len(tokenizer))
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False)
assert 'model' in checkpoint and 'step' in checkpoint, 'Formato de checkpoint inesperado.'
print('Step do checkpoint:',checkpoint['step'])
assert int(checkpoint['step'])==300000, 'Checkpoint não é step 300000: revise antes de continuar.'
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint; gc.collect()
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()
model=model.to(device='cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)
model.eval()
print('Modelo carregado, pesos base congelados.')


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)

for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,RANK,ALPHA,LORA_DROPOUT)
    block.proj=LoRALinear(block.proj,RANK,ALPHA,LORA_DROPOUT)
trainable=[p for p in model.parameters() if p.requires_grad]
print('LoRA trainable parameters:',sum(p.numel() for p in trainable))
assert len(trainable)==cfg.n_layer*2*2


## Preparação: separador explícito, offsets e máscara causal

Cada amostra é codificada como `prompt + '\n' + completion + EOS`. O **limite da resposta** é medido em offsets de caracteres do texto concatenado; tokens que cruzam a fronteira são rejeitados (esperado: zero). Os labels são deslocados em um token e a loss incide somente na resposta e EOS. O corpus permanece imutável.

In [ ]:
def read_jsonl(path):
    with open(path,encoding='utf-8') as f:return [json.loads(line) for line in f if line.strip()]
train_raw=read_jsonl(TRAIN_FILE);val_raw=read_jsonl(VAL_FILE)
assert len(train_raw)==29995 and len(val_raw)==1521,(len(train_raw),len(val_raw))
assert not ({r['paperId'] for r in train_raw if r.get('paperId')}&{r['paperId'] for r in val_raw if r.get('paperId')})
assert not ({r['groupId'] for r in train_raw if r.get('groupId')}&{r['groupId'] for r in val_raw if r.get('groupId')})
print('Splits íntegros, sem sobreposição de paperId/groupId.')

def encode_record(rec):
    prompt=str(rec['prompt']);answer=str(rec['completion'])
    if not prompt.strip() or not answer.strip():return None,'empty'
    full_prompt=prompt+SEPARATOR
    text=full_prompt+answer
    enc=tokenizer(text,add_special_tokens=False,return_offsets_mapping=True)
    ids=enc['input_ids'];offsets=enc['offset_mapping'];boundary=len(full_prompt)
    if any(a<boundary<b for a,b in offsets):return None,'boundary_merge'
    answer_idx=next((i for i,(a,b) in enumerate(offsets) if a>=boundary and b>a),None)
    if answer_idx is None:return None,'no_answer_tokens'
    ids=ids+[tokenizer.eos_token_id]
    if len(ids)>MAX_LENGTH:return None,'too_long'
    labels=[-100]*(len(ids)-1)
    for i in range(len(labels)):
        if i+1>=answer_idx:labels[i]=ids[i+1]
    if all(v==-100 for v in labels):return None,'empty_targets'
    return {'x':ids[:-1],'y':labels,'language':rec['language'],'id':rec['id'],
            'answer_idx':answer_idx,'prompt_text':full_prompt},None

def prepare(rows):
    good=[];dropped={}
    for r in tqdm(rows,desc='Tokenização'):
        item,why=encode_record(r)
        if item is not None:good.append(item)
        else:dropped[why]=dropped.get(why,0)+1
    return good,dropped
train_data,train_drop=prepare(train_raw);val_data,val_drop=prepare(val_raw)
print('Train:',len(train_data),'excluídos:',train_drop)
print('Val:',len(val_data),'excluídos:',val_drop)
assert train_data and val_data
assert train_drop.get('boundary_merge',0)==0 and val_drop.get('boundary_merge',0)==0
for name,data in [('train',train_data),('validation',val_data)]:
    print(name,'idiomas:',{lang:sum(r['language']==lang for r in data) for lang in ['pt','en','es']})

In [ ]:
def collate(batch):
    L=max(len(r['x']) for r in batch)
    x=torch.full((len(batch),L),tokenizer.eos_token_id,dtype=torch.long)
    y=torch.full((len(batch),L),-100,dtype=torch.long)
    for i,r in enumerate(batch):
        l=len(r['x']);x[i,:l]=torch.tensor(r['x']);y[i,:l]=torch.tensor(r['y'])
    return x.to('cuda'),y.to('cuda')

# Verificação de alinhamento em exemplos PT/EN/ES, incluindo EOS
for lang in ('pt','en','es'):
    sample=next(r for r in train_data if r['language']==lang)
    assert len(sample['x'])==len(sample['y'])
    mask=[i for i,z in enumerate(sample['y']) if z!=-100]
    assert mask and mask[0]==sample['answer_idx']-1
    assert all(sample['y'][i]==-100 for i in range(mask[0]))
    assert sample['y'][-1]==tokenizer.eos_token_id
    print(lang,'tokens:',len(sample['x']),'supervisionados:',len(mask),
          'resposta:',tokenizer.decode([sample['y'][i] for i in mask])[:150])

# Amostra longa para testar pico de VRAM sem otimização
if RUN_MEMORY_PROBE:
    model.train();optimizer_probe_grad_before=sum(p.grad is not None for p in trainable)
    assert optimizer_probe_grad_before==0
    probe=max(train_data,key=lambda r:len(r['x']))
    torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    try:
        x,y=collate([probe])
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
        loss=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100)
        loss.backward()
        torch.cuda.synchronize()
        print('Smoke test OK | tokens:',len(probe['x']),
              '| peak VRAM GiB:',round(torch.cuda.max_memory_allocated()/2**30,2),
              '| total GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
    finally:
        for p in trainable:p.grad=None
        for v in ('x','y','logits','loss'):
            if v in globals():del globals()[v]
        model.eval();gc.collect();torch.cuda.empty_cache()

## Validação, configuração e checkpoints

A validação é ponderada por **tokens supervisionados**, com perdas separadas por PT/EN/ES. Checkpoints contêm LoRA, optimizer, scheduler, RNG Python/NumPy/PyTorch/CUDA e hash de dados. `adapter_best.pt` contém apenas pesos LoRA + metadados. A retomada parte exclusivamente de um checkpoint completo, nunca de um update interrompido.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
DATA_HASHES={'train':sha256(TRAIN_FILE),'validation':sha256(VAL_FILE)}

def lora_state():
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items() if '.A' in k or '.B' in k}

def restore_lora(state):
    keys={k for k in model.state_dict() if '.A' in k or '.B' in k}
    assert set(state)==keys,(len(state),len(keys))
    with torch.no_grad():
        for name,param in model.named_parameters():
            if name in state:param.copy_(state[name].to(param.device,dtype=param.dtype))

@torch.no_grad()
def evaluate():
    was_training=model.training;model.eval(); sums={}; counts={}
    for lang in ['pt','en','es']:
        sums[lang]=0.;counts[lang]=0
    for start in range(0,len(val_data),MICRO_BATCH):
        batch=val_data[start:start+MICRO_BATCH]
        x,y=collate(batch)
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
        losses=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100,reduction='none').reshape_as(y)
        for i,r in enumerate(batch):
            mask=y[i]!=-100;lang=r['language']
            sums[lang]+=float(losses[i][mask].sum());counts[lang]+=int(mask.sum())
        del x,y,logits,losses
    result={f'val_loss_{k}':sums[k]/max(counts[k],1) for k in sums}
    result['val_loss']=sum(sums.values())/max(sum(counts.values()),1)
    model.train(was_training);return result

optimizer=torch.optim.AdamW(trainable,lr=LR,weight_decay=WEIGHT_DECAY)
updates_per_epoch=math.ceil(math.ceil(len(train_data)/MICRO_BATCH)/GRAD_ACCUM)
planned_updates=updates_per_epoch*EPOCHS
if MAX_STEPS is not None:planned_updates=min(planned_updates,MAX_STEPS)
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s: max(0.1,1-s/max(planned_updates,1)))
print('Updates/epoch:',updates_per_epoch,'planned:',planned_updates)

In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
CONFIG={'experiment':'sft_v2_3_lora_v1','rank':RANK,'alpha':ALPHA,'dropout':LORA_DROPOUT,
        'lr':LR,'weight_decay':WEIGHT_DECAY,'max_length':MAX_LENGTH,'separator':SEPARATOR,
        'micro_batch':MICRO_BATCH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,
        'max_steps':MAX_STEPS,'max_grad_norm':MAX_GRAD_NORM,'seed':SEED,
        'data_hashes':DATA_HASHES,'checkpoint_path':str(BASE_CKPT),'base_step':300000,
        'target_modules':['qkv','proj'],'architecture':'MaiaGPT v3.8',
        'validate_every':VALIDATE_EVERY,'save_every':SAVE_EVERY,
        'torch_version':torch.__version__,'gpu':torch.cuda.get_device_name(0)}
config_file=OUT/'config.json'
if config_file.exists():
    old=json.loads(config_file.read_text())
    assert old==CONFIG,'Configuração/ambiente mudou; use novo OUT ou restaure configuração original.'
else:
    config_file.write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False))
ckpt_file=OUT/'adapter_latest.pt';start_update=0
if ckpt_file.exists():
    saved=torch.load(ckpt_file,map_location='cpu',weights_only=False)
    assert saved['config']==CONFIG
    restore_lora(saved['adapter'])
    optimizer.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler'])
    start_update=int(saved['update'])
    random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng'])
    torch.set_rng_state(saved['torch_rng'])
    torch.cuda.set_rng_state_all(saved['cuda_rng'])
    print('Retomada do último checkpoint COMPLETO:',start_update)
    del saved
else:print('Novo experimento v2.3, sem adaptador anterior.')

def save_adapter(update,metrics=None):
    state={'adapter':lora_state(),'optimizer':optimizer.state_dict(),
           'scheduler':scheduler.state_dict(),'update':update,'config':CONFIG,'metrics':metrics,
           'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
           'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
    tmp=OUT/'adapter_latest.tmp.pt'
    torch.save(state,tmp);os.replace(tmp,ckpt_file)
    print('Checkpoint íntegro salvo:',ckpt_file,'update:',update)

BEST_META=OUT/'best_validation.json'
best_loss=float(json.loads(BEST_META.read_text())['val_loss']) if BEST_META.exists() else float('inf')
def maybe_save_best(update,metrics):
    global best_loss
    if metrics and 'val_loss' in metrics and metrics['val_loss']<best_loss:
        best_loss=float(metrics['val_loss'])
        state={'adapter':lora_state(),'update':update,'val_loss':best_loss,'config':CONFIG}
        temp=OUT/'adapter_best.tmp.pt';torch.save(state,temp)
        os.replace(temp,OUT/'adapter_best.pt')
        BEST_META.write_text(json.dumps({'update':update,'val_loss':best_loss},indent=2))
        print('Melhor adaptador v2.3:',update,best_loss)

## Execução

Com `RUN_TRAINING=False`, realiza somente pré-avaliação. Com `True`, inicia um experimento **novo** a partir da base ou retoma o último checkpoint completo. Em caso de interrupção durante um update, **não salva gradientes incompletos**: a próxima execução retoma o último checkpoint periódico salvo, sem fingir que o update foi concluído. Para preservar a trajetória de RNG, não execute testes de geração estocásticos entre restaurar o checkpoint e retomar o loop.

In [ ]:
if not RUN_TRAINING:
    print('Treinamento DESATIVADO; pré-avaliação:',evaluate())
    print('Para iniciar, defina RUN_TRAINING=True na célula de parâmetros e execute novamente desde o início.')
else:
    assert start_update<=planned_updates
    model.train();optimizer.zero_grad(set_to_none=True)
    update=start_update
    progress=tqdm(total=planned_updates,initial=update,desc='SFT v2.3 LoRA')
    log_path=OUT/'metrics.jsonl'
    try:
        for epoch in range(EPOCHS):
            if update>=planned_updates:break
            order=np.random.default_rng(SEED+epoch).permutation(len(train_data)).tolist()
            micro_batches=[order[i:i+MICRO_BATCH] for i in range(0,len(order),MICRO_BATCH)]
            n_updates=math.ceil(len(micro_batches)/GRAD_ACCUM)
            for local_update in range(n_updates):
                global_update=epoch*updates_per_epoch+local_update
                if global_update<update:continue
                if update>=planned_updates:break
                indices=micro_batches[local_update*GRAD_ACCUM:(local_update+1)*GRAD_ACCUM]
                total_tokens=sum(sum(z!=-100 for z in train_data[j]['y']) for ids in indices for j in ids)
                accumulated_loss=0.
                for ids in indices:
                    x,y=collate([train_data[j] for j in ids])
                    with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
                    loss_sum=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),
                                             y.reshape(-1),ignore_index=-100,reduction='sum')
                    (loss_sum/max(total_tokens,1)).backward()
                    accumulated_loss+=float(loss_sum.detach())
                    del x,y,logits,loss_sum
                grad_norm=torch.nn.utils.clip_grad_norm_(trainable,MAX_GRAD_NORM)
                optimizer.step();scheduler.step();optimizer.zero_grad(set_to_none=True)
                update+=1;progress.update(1)
                metrics={'update':update,'epoch':epoch+1,'train_loss':accumulated_loss/max(total_tokens,1),
                         'grad_norm':float(grad_norm),'lr':optimizer.param_groups[0]['lr']}
                if update%VALIDATE_EVERY==0 or update==planned_updates:
                    metrics.update(evaluate());print(metrics);maybe_save_best(update,metrics)
                with open(log_path,'a',encoding='utf-8') as f:f.write(json.dumps(metrics)+'\n')
                if update%SAVE_EVERY==0 or update==planned_updates:save_adapter(update,metrics)
    except KeyboardInterrupt:
        optimizer.zero_grad(set_to_none=True)
        print('Interrompido. Próxima execução retomará SOMENTE o último checkpoint salvo em:',ckpt_file)
        print('NÃO foi salvo estado parcial do update',update)
        raise
    finally:progress.close()
    print('Treinamento finalizado até update',update)

## Avaliação qualitativa após o treinamento

A célula seguinte serve apenas para inspeção de respostas e **não deve ser executada antes de uma retomada determinística**. A loss global é medida no mesmo conjunto de validação v2.2, mas o novo separador/contexto alteram o conjunto de exemplos aceitos; por isso não compare perdas brutas v2.2/v2.3 sem uma avaliação comum pareada.

In [ ]:
@torch.no_grad()
def generate(prompt,max_new_tokens=100,temperature=0.7):
    model.eval()
    ids=tokenizer(prompt,add_special_tokens=False,return_tensors='pt')['input_ids'].to('cuda')
    for _ in range(max_new_tokens):
        if ids.shape[1]>=cfg.block_size:break
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(ids)
        last=logits[:,-1,:].float()/max(temperature,1e-4)
        nxt=torch.multinomial(torch.softmax(last,dim=-1),1)
        if nxt.item()==tokenizer.eos_token_id:break
        ids=torch.cat([ids,nxt],dim=1)
    model.train()
    return tokenizer.decode(ids[0],skip_special_tokens=True)

for lang in ['pt','en','es']:
    r=next(r for r in val_raw if r['language']==lang)
    print('\n',lang,r['id'],'\n',generate(r['prompt'])[-1200:])

## Critérios de aceitação e próximos passos

1. Auditoria de hashes aprovada; corpus sem alteração.
2. `boundary_merge=0`; conferir cobertura para 768/1024 e teste causal PT/EN/ES.
3. Smoke test de VRAM sem OOM; 1024 só se houver margem.
4. Treino em novo diretório `...sft_v2_3_lora_v1`, sem sobrescrever v2.2.
5. `adapter_latest.pt` e `adapter_best.pt` gerados; validação por idioma.
6. Após a primeira época, comparar geração com v2.2 e modelo-base usando **mesmo conjunto de prompts e decodificação**; não concluir superioridade apenas pela loss.

**Limitação de retomada:** o estado RNG/optimizer é salvo em checkpoints periódicos; retomar do último checkpoint completo é reprodutível no mesmo ambiente quando kernels determinísticos permitem, mas a equivalência bit a bit ainda exige um teste de interrupção e comparação de hashes de pesos. Não declarar tal teste aprovado sem executá-lo.